# **HƯỚNG DẪN HUẤN LUYỆN MÔ HÌNH MT5 TRÊN GOOGLE COLAB**
Chào mừng bạn đến với Notebook hướng dẫn huấn luyện mô hình tóm tắt văn bản tiếng Việt **mT5** (`google/mt5-small` của Google).
Notebook này được thiết kế và tối ưu để chạy hoàn toàn miễn phí trên **Google Colab (sử dụng GPU T4)**.

**Đặc trưng của mT5:** Tương tự như ViT5, mô hình này được huấn luyện với cấu trúc T5 và yêu cầu thêm tiền tố `summarize: ` vào trước văn bản đầu vào.

**Bộ dữ liệu mục tiêu:** `8Opt/vietnamese-summarization-dataset-0001` (Gồm 19,525 mẫu dữ liệu tóm tắt tiếng Việt sạch).

---

## **Bước 1: Bật GPU trên Google Colab**
Để huấn luyện với tốc độ nhanh nhất, bạn cần kích hoạt bộ tăng tốc phần cứng GPU:
1. Trên thanh công cụ phía trên, chọn **Runtime** (Thời gian chạy) $\rightarrow$ **Change runtime type** (Thay đổi loại thời gian chạy).
2. Tại mục **Hardware accelerator** (Bộ tăng tốc phần cứng), chọn **T4 GPU**.
3. Nhấn **Save** (Lưu).
4. Chạy ô mã nguồn dưới đây để kiểm tra thông tin card đồ họa của bạn.

In [ ]:
!nvidia-smi

## **Bước 2: Cài đặt các thư viện cần thiết**
Chúng ta cài đặt thư viện Transformers và các công cụ bổ trợ.

In [ ]:
!pip install -q transformers[torch]>=4.38.0 accelerate>=0.27.0 datasets>=2.18.0 sentencepiece>=0.1.99 underthesea>=6.8.0 pyvi>=0.1.1 rouge-score>=0.1.2 bert-score>=0.3.13 sentence-transformers>=3.0.0 pandas pyarrow

## **Bước 3: Chuẩn bị Dữ liệu Huấn luyện**
Notebook hỗ trợ nạp dữ liệu tự động từ Hugging Face hoặc đọc dữ liệu parquet tải lên thủ công.

In [ ]:
import os
import glob
from datasets import Dataset, DatasetDict, load_dataset

DATASET_HF = "8Opt/vietnamese-summarization-dataset-0001"
parquet_dir = "/content/vietnamese-summarization-dataset-0001"

if os.path.exists(parquet_dir) and glob.glob(os.path.join(parquet_dir, "*.parquet")):
    print("\u2705 Tìm thấy dữ liệu Parquet local trên Colab! Đang tải dữ liệu...")
    train_files = sorted(glob.glob(os.path.join(parquet_dir, "train*.parquet")))
    val_files = sorted(glob.glob(os.path.join(parquet_dir, "validation*.parquet")) or glob.glob(os.path.join(parquet_dir, "val*.parquet")))
    test_files = sorted(glob.glob(os.path.join(parquet_dir, "test*.parquet")))
    
    dataset = DatasetDict({
        "train": Dataset.from_parquet(train_files[0]) if train_files else None,
        "validation": Dataset.from_parquet(val_files[0]) if val_files else None,
        "test": Dataset.from_parquet(test_files[0]) if test_files else None,
    })
    
    # Nếu thiếu validation, thực hiện split tự động từ tập train
    if dataset["validation"] is None and dataset["train"] is not None:
        print("Tập Validation trống, tự động tách 10% từ tập Train làm Validation...")
        split = dataset["train"].train_test_split(test_size=0.1, seed=42)
        dataset = DatasetDict({"train": split["train"], "validation": split["test"]})
else:
    print(f"\u2139\ufe0f Không thấy dữ liệu Parquet local. Tự động tải bộ dữ liệu '{DATASET_HF}' trực tiếp từ Hugging Face...")
    raw = load_dataset(DATASET_HF)
    dataset = DatasetDict({
        "train": raw.get("train") or raw.get("train-00000-of-00001"),
        "validation": raw.get("validation") or raw.get("validation-00000-of-00001") or raw.get("val"),
        "test": raw.get("test") or raw.get("test-00000-of-00001")
    })
    
    # Dự phòng nếu chia tập không đầy đủ
    if dataset["validation"] is None and dataset["train"] is not None:
        print("Tập Validation trống, tự động tách 10% từ tập Train làm Validation...")
        split = dataset["train"].train_test_split(test_size=0.1, seed=42)
        dataset = DatasetDict({"train": split["train"], "validation": split["test"]})

print(f"\u2728 Tải dữ liệu thành công!")
print(f" - Số lượng mẫu Train: {len(dataset['train']):,}")
print(f" - Số lượng mẫu Validation: {len(dataset['validation']):,}")

## **Bước 4: Tiền xử lý & Tokenize cho mT5**
mT5 yêu cầu thêm tiền tố `summarize: ` vào trước văn bản đầu vào.

In [ ]:
import torch
from transformers import T5Tokenizer

MODEL_NAME = "google/mt5-small"
MAX_INPUT_TOKENS = 512
MAX_TARGET_TOKENS = 128

print(f"\ud83d\udd04 Đang khởi tạo Tokenizer cho mT5 từ: {MODEL_NAME}...")
# Sử dụng trực tiếp T5Tokenizer để tránh lỗi convert_to_native_format trong Python 3.12
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

# Tự động phát hiện cột dữ liệu
available_cols = dataset["train"].column_names
input_col = next((col for col in ["document", "article", "text", "content"] if col in available_cols), None)
target_col = next((col for col in ["summary", "title", "headline"] if col in available_cols), None)

if not input_col or not target_col:
    raise ValueError(f"Không thể nhận diện cột văn bản gốc và nhãn tóm tắt từ danh sách: {available_cols}")

print(f"\u2705 Tự động ánh xạ thành công:")
print(f" - Cột văn bản gốc đầu vào (Input): '{input_col}'")
print(f" - Cột nhãn tóm tắt đầu ra (Target): '{target_col}'")

def _prefix(text: str) -> str:
    return f"summarize: {text}"

def tokenize_batch(examples):
    inputs = [_prefix(str(article)) for article in examples[input_col]]
    targets = [str(summary) for summary in examples[target_col]]
    
    model_inputs = tokenizer(
        inputs, 
        max_length=MAX_INPUT_TOKENS, 
        truncation=True, 
        padding=False
    )
    labels = tokenizer(
        text_target=targets, 
        max_length=MAX_TARGET_TOKENS, 
        truncation=True, 
        padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("\ud83d\udcd8 Đang tiến hành phân đoạn từ (Tokenizing) dữ liệu...")
train_cols = dataset["train"].column_names
tokenized = dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=64,
    remove_columns=train_cols,
    desc="Processing"
)
print("\u2705 Tiền xử lý dữ liệu hoàn thành!")

## **Bước 5: Thiết lập Cấu hình & Huấn luyện mT5**
Cấu hình tối ưu cao cho GPU T4 VRAM 16GB.

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)

print(f"\ud83d\udce5 Đang tải mô hình gốc: {MODEL_NAME}...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-colab-checkpoints",
    num_train_epochs=3,                     
    per_device_train_batch_size=8,          
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,          
    learning_rate=5e-5,                     
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="steps",
    eval_steps=200,                         
    save_strategy="steps",
    save_steps=200,                         
    save_total_limit=2,                     
    load_best_model_at_end=True,            
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,                              
    logging_steps=50,
    report_to="none",
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,                       
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\ud83d\ude80 Đang bắt đầu tiến hành Huấn Luyện mT5...")
trainer.train()
print("\u2705 Quá trình huấn luyện đã kết thúc xuất sắc!")

trainer.save_model("./mt5-colab-finetuned")
tokenizer.save_pretrained("./mt5-colab-finetuned")
print("\ud83d\udcbe Đã lưu trữ trọng số mô hình tốt nhất vào thư mục: ./mt5-colab-finetuned")

## **Bước 6: Tải Mô hình về Máy Local**

In [ ]:
print("\ud83d\udce6 Đang tiến hành nén mô hình thành file zip...")
!zip -r mt5-colab-finetuned.zip ./mt5-colab-finetuned
print("\u2705 Đã nén thành công file: mt5-colab-finetuned.zip")

### **Lựa chọn 1: Lưu trữ file Zip lên Google Drive cá nhân (Khuyên dùng)**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sao chép file zip sang Google Drive của bạn
!cp mt5-colab-finetuned.zip /content/drive/MyDrive/
print("\u2705 Đã lưu trữ thành công file 'mt5-colab-finetuned.zip' vào Google Drive của bạn!")

### **Lựa chọn 2: Tải xuống trực tiếp qua trình duyệt web**

In [ ]:
from google.colab import files
files.download("mt5-colab-finetuned.zip")